# 01 — Data Acquisition
**Steps 3–11:** GEE authentication, ROI definition, Sentinel-2 & Sentinel-1 composites, label reclassification, and export.

> ⚠️ Requires a Google Earth Engine account. Run `earthengine authenticate` in terminal before this notebook.

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import ee
import geemap
import os

# ── Authenticate & Initialise GEE ───────────────────────────
try:
    ee.Initialize()
    print('✅ GEE initialised.')
except Exception:
    ee.Authenticate()
    ee.Initialize()
    print('✅ GEE authenticated and initialised.')

# ── Paths ────────────────────────────────────────────────────
RAW_DIR = os.path.join('..', 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)

In [ ]:
# ── STEP 4: Define Region of Interest ───────────────────────
# Option A — draw on map
Map = geemap.Map(center=[6.2, -1.67], zoom=11)
Map

# Option B — load boundary shapefile / GEE asset
# ROI = ee.FeatureCollection('projects/YOUR_PROJECT/assets/obuasi_boundary').geometry()

# Placeholder bounding box (replace with actual Obuasi boundary)
ROI = ee.Geometry.Rectangle([-1.75, 6.15, -1.60, 6.25])
print('ROI defined.')

In [ ]:
# ── STEPS 5–7: Sentinel-2 Annual Composites ──────────────────

S2_BANDS  = ['B2', 'B3', 'B4', 'B8', 'B11']
YEARS     = list(range(2018, 2026))

def mask_s2_clouds(image):
    """Mask clouds using Sentinel-2 QA60 band."""
    qa    = image.select('QA60')
    cloud = qa.bitwiseAnd(1 << 10).eq(0)   # opaque clouds
    cirrus = qa.bitwiseAnd(1 << 11).eq(0)  # cirrus
    return image.updateMask(cloud.And(cirrus)).divide(10000)  # scale to [0,1]

def get_s2_composite(year, roi):
    """Annual median composite from Sentinel-2 SR Harmonized."""
    col = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_s2_clouds)
        .select(S2_BANDS)
        .median()
        .clip(roi)
    )
    return col

# Build composites for all years
s2_composites = {year: get_s2_composite(year, ROI) for year in YEARS}
print(f'✅ Sentinel-2 composites prepared for years: {YEARS}')

In [ ]:
# ── STEP 6: Sentinel-1 Radar Features ────────────────────────

def get_s1_composite(year, roi):
    """Annual median composite from Sentinel-1 GRD with VV/VH ratio."""
    col = (
        ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .select(['VV', 'VH'])
        .median()
        .clip(roi)
    )
    # Compute VV/VH ratio in dB: ratio = VV - VH
    vv_vh_ratio = col.select('VV').subtract(col.select('VH')).rename('VV_VH_ratio')
    return col.addBands(vv_vh_ratio)

s1_composites = {year: get_s1_composite(year, ROI) for year in YEARS}
print(f'✅ Sentinel-1 composites prepared for years: {YEARS}')

In [ ]:
# ── STEP 8: Form Multimodal Datacube ─────────────────────────
# Stack S2 + S1 into an 8-band image per year

datacubes = {}
for year in YEARS:
    datacubes[year] = (
        s2_composites[year]
        .addBands(s1_composites[year])
        .rename(['B2', 'B3', 'B4', 'B8', 'B11', 'VV', 'VH', 'VV_VH_ratio'])
    )

print('✅ 8-band multimodal datacubes formed for all years.')
print('   Bands: B2, B3, B4, B8, B11, VV, VH, VV_VH_ratio')

In [ ]:
# ── STEP 10: Prepare Reference Labels (ESRI 10m Land Cover) ──

# ESRI class → project class mapping
# ESRI:  1=Water, 2=Trees, 4=Flooded Veg, 5=Crops, 7=Built Area, 8=Bare Ground,
#        9=Snow/Ice, 10=Clouds, 11=Rangeland
ESRI_TO_PROJECT = {
    1:  1,   # Water
    2:  2,   # Trees
    5:  3,   # Crops
    7:  4,   # Built-up
    8:  5,   # Bare Ground
    4:  3,   # Flooded Veg → Crops (approximate)
    11: 3,   # Rangeland → Crops (approximate)
}

def reclassify_esri(year, roi):
    """Load ESRI annual land cover and reclassify to 5-class scheme."""
    esri = (
        ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS')
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .mosaic()
        .clip(roi)
    )

    # Reclassify
    from_vals = list(ESRI_TO_PROJECT.keys())
    to_vals   = list(ESRI_TO_PROJECT.values())
    return esri.remap(from_vals, to_vals, defaultValue=0)

# Use 2025 as the primary reference label for training/eval
label_2025 = reclassify_esri(2025, ROI)
print('✅ ESRI labels reclassified into 5-class scheme for 2025.')

In [ ]:
# ── STEP 11: Export Datacubes and Labels to Google Drive ─────

EXPORT_SCALE  = 10   # 10 m resolution
EXPORT_CRS    = 'EPSG:32630'   # UTM Zone 30N — appropriate for Ghana
DRIVE_FOLDER  = 'Obuasi_GEE_Exports'

def export_image(image, description, folder, roi, scale=10, crs='EPSG:32630'):
    """Submit a GEE export task to Google Drive."""
    task = ee.batch.Export.image.toDrive(
        image         = image,
        description   = description,
        folder        = folder,
        fileNamePrefix= description,
        region        = roi,
        scale         = scale,
        crs           = crs,
        maxPixels     = 1e13,
        fileFormat    = 'GeoTIFF'
    )
    task.start()
    return task

export_tasks = []

# Export 2025 datacube + labels (needed for training)
export_tasks.append(export_image(datacubes[2025], 'datacube_2025', DRIVE_FOLDER, ROI))
export_tasks.append(export_image(label_2025, 'labels_2025', DRIVE_FOLDER, ROI))

# Export all annual datacubes for change analysis
for year in YEARS:
    export_tasks.append(export_image(datacubes[year], f'datacube_{year}', DRIVE_FOLDER, ROI))

print(f'✅ {len(export_tasks)} export tasks submitted to Google Drive folder: {DRIVE_FOLDER}')
print('   Monitor progress at: https://code.earthengine.google.com/tasks')

In [ ]:
# ── Check task status ─────────────────────────────────────────
for task in export_tasks[:5]:   # show first 5
    print(f'  {task.id} — {task.status()["state"]}')